<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 02. Clasificación Multiclase y Fronteras de Decisión
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 08
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/08%20-%20Classification/Para%20Dummies/02_Clasificacion_Multiclase_y_Fronteras_Decision_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## 🗳️ La Analogía del Concurso de Talento (más de 2 opciones)

Hasta ahora vimos modelos que deciden entre **2 opciones** (sí/no, enfermo/sano). Pero, ¿qué pasa cuando hay **3 o más categorías**? Por ejemplo: decidir si una flor es *Setosa*, *Versicolor* o *Virginica*.

Imagina un concurso de talento con 3 concursantes. Hay dos formas de decidir un ganador:

* **One-vs-Rest (OvR — "Uno contra el Resto"):** se arman 3 jurados independientes. El jurado 1 solo pregunta *"¿Es el concursante A el mejor, sí o no?"*, el jurado 2 pregunta lo mismo del concursante B, y el jurado 3 del concursante C. Gana el concursante cuyo jurado quedó **más convencido**.
* **Softmax / Multinomial:** hay un **único comité** que reparte 100 puntos de probabilidad entre los 3 concursantes a la vez (por ejemplo: 70% A, 20% B, 10% C), y siempre suman 100%.

Scikit-Learn puede usar cualquiera de las dos estrategias. Nosotros usaremos la versión Softmax, la más usada en la práctica.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data[:, [2, 3]]  # solo longitud y ancho del petalo, para poder dibujar en 2D
y = iris.target

df_muestra = pd.DataFrame(X, columns=['longitud_petalo', 'ancho_petalo'])
df_muestra['especie'] = [iris.target_names[i] for i in y]

print("Especies posibles:", list(iris.target_names))
print("Cantidad de muestras por especie:")
print(df_muestra['especie'].value_counts())
df_muestra.head()


### 🤔 ¿Qué acaba de pasar?

- Cargamos el famoso dataset **Iris**: 150 flores repartidas en partes iguales entre 3 especies (50 de cada una).
- Nos quedamos solo con **2 medidas** (longitud y ancho del pétalo) para poder dibujar todo en un plano 2D — con las 4 medidas originales no cabría en una hoja de papel.
- `y` guarda un número (0, 1 o 2) por cada flor: esa es la **etiqueta** que el modelo debe aprender a predecir. `0` = setosa, `1` = versicolor, `2` = virginica.


---
## 🗺️ Dibujando el Mapa de Territorios

Piensa en un mapa político: cada país tiene un color distinto y hay líneas (fronteras) que separan un territorio de otro. Un modelo de clasificación hace lo mismo con los datos: pinta el plano en "territorios" (uno por especie) y traza **fronteras de decisión** — las líneas imaginarias donde el modelo cambia de opinión sobre a qué especie pertenece una flor.

Vamos a entrenar el modelo Softmax y a pintar su mapa de territorios.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

modelo_multiclase = LogisticRegression(multi_class='multinomial', max_iter=1000)
modelo_multiclase.fit(X, y)

# Creamos una cuadricula fina que cubre todo el plano, para "pintar" cada territorio
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))
Z = modelo_multiclase.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(7, 5))
plt.contourf(xx, yy, Z, alpha=0.35, cmap='viridis')
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', edgecolor='k', s=50)
plt.title('Territorios de cada especie segun el modelo', fontweight='bold')
plt.xlabel('Longitud del petalo (cm)')
plt.ylabel('Ancho del petalo (cm)')
plt.show()


### 🤔 ¿Qué acaba de pasar?

- Cada color de fondo es el **territorio** que el modelo asigna a una especie: si un punto cae en la zona morada, el modelo dice "esto es setosa".
- Los puntos son las flores reales, coloreadas según su especie verdadera. Cuando un punto cae en la zona de su mismo color, el modelo acertó.
- Las líneas donde un color termina y empieza otro son las **fronteras de decisión**. Con solo 2 medidas y un modelo lineal, esas fronteras son casi rectas.


---
## 🌀 Cuando una Línea Recta No Alcanza

Imagina que tienes canicas rojas y azules acomodadas en forma de espiral, y te piden separarlas usando solo una **cuerda recta**. Es imposible: por más que muevas la cuerda, siempre quedarán canicas del color equivocado de un lado.

La solución es dejar que la cuerda se **doble**. Eso es justo lo que hace `PolynomialFeatures`: en lugar de darle al modelo solo `longitud` y `ancho`, también le da combinaciones como `longitud²`, `ancho²` y `longitud × ancho`. Con esas "curvas extra" disponibles, el modelo puede doblar sus fronteras para adaptarse a formas más complicadas.


In [ ]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline

modelo_curvo = Pipeline([
    ('escalador', StandardScaler()),
    ('curvas', PolynomialFeatures(degree=3, include_bias=False)),
    ('clasificador', LogisticRegression(max_iter=2000))
])
modelo_curvo.fit(X, y)

Z_curvo = modelo_curvo.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(7, 5))
plt.contourf(xx, yy, Z_curvo, alpha=0.35, cmap='viridis')
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', edgecolor='k', s=50)
plt.title('Territorios con fronteras curvas (polinomio grado 3)', fontweight='bold')
plt.xlabel('Longitud del petalo (cm)')
plt.ylabel('Ancho del petalo (cm)')
plt.show()


### 🤔 ¿Qué acaba de pasar?

- `StandardScaler` pone las dos medidas en la misma escala antes de combinarlas (como convertir todo a los mismos "centímetros justos").
- `PolynomialFeatures(degree=3)` fabrica automáticamente esas combinaciones curvas que mencionamos.
- Compara este mapa con el anterior: las fronteras ya no son líneas rectas, sino curvas que envuelven mejor a cada especie.


---
## 🧳 La Maleta con Límite de Peso (Regularización)

Imagina que vas a viajar y la aerolínea te limita a **20 kg de equipaje**. Tienes que decidir qué llevar y qué dejar en casa. Hay dos formas de empacar:

* **Regularización L1 (Lasso):** eres estricto — si algo no es esencial, lo **dejas completamente afuera** de la maleta (su "peso", o coeficiente, queda en exactamente 0). El resultado es una maleta ligera con solo lo importante.
* **Regularización L2 (Ridge):** en cambio, metes todo, pero comprimes un poco cada cosa para que quepa (ningún coeficiente llega a 0, todos se reducen un poco).

L1 es especialmente útil cuando sospechas que **algunas características no aportan nada** — el modelo mismo se encarga de descartarlas.


In [ ]:
X_completo = iris.data  # ahora usamos las 4 medidas: sepalo y petalo
modelo_l1 = LogisticRegression(penalty='l1', solver='saga', C=0.3, max_iter=3000)
modelo_l1.fit(X_completo, y)

tabla_coeficientes = pd.DataFrame(
    modelo_l1.coef_,
    columns=iris.feature_names,
    index=iris.target_names
)
print("Coeficientes con la maleta apretada (L1). Los valores en 0 son caracteristicas descartadas:")
tabla_coeficientes.round(2)


### 🤔 ¿Qué acaba de pasar?

- Entrenamos el modelo con `penalty='l1'`, la versión "maleta estricta". El parámetro `C` controla qué tan estricta es: **entre más pequeño `C`, más estricta la maleta** (más ceros aparecen).
- La tabla muestra un coeficiente por cada combinación de especie y característica. Si ves un `0.0`, significa que el modelo decidió que esa medida **no le sirve** para distinguir esa especie de las demás.
- Esto es una forma automática de "selección de características": el modelo te dice cuáles medidas realmente importan.


---
##### 🎯 Reto Práctico para Dummies: Una Maleta Más Floja

**Situación:** Ahora prueba con una maleta menos estricta, usando `C=5.0` en lugar de `C=0.3`.

1. Entrena un nuevo modelo `LogisticRegression(penalty='l1', solver='saga', C=5.0, max_iter=3000)` con `X_completo` y `y`.
2. Cuenta cuántos coeficientes quedaron exactamente en 0 (usa `np.sum(modelo.coef_ == 0)`).
3. Compáralo con los ceros que obtuvo el modelo con `C=0.3`. ¿Hay más o menos características descartadas?


In [ ]:
# =========================================================================
# TU SOLUCION: Reto Dummies 2 - Maleta mas floja (C=5.0)
# =========================================================================

# 1. Entrenar el nuevo modelo
# modelo_l1_flojo = LogisticRegression(penalty='l1', solver='saga', C=5.0, max_iter=3000)
# modelo_l1_flojo.fit(X_completo, y)

# 2. Contar ceros
# ceros_flojo = np.sum(modelo_l1_flojo.coef_ == 0)
# ceros_estricto = np.sum(modelo_l1.coef_ == 0)


<details>
<summary><b>💡 Haz clic aquí para ver la solución explicada...</b></summary>

```python
modelo_l1_flojo = LogisticRegression(penalty='l1', solver='saga', C=5.0, max_iter=3000)
modelo_l1_flojo.fit(X_completo, y)

ceros_flojo = np.sum(modelo_l1_flojo.coef_ == 0)
ceros_estricto = np.sum(modelo_l1.coef_ == 0)

print(f"🧳 Maleta estricta (C=0.3): {ceros_estricto} coeficientes en cero")
print(f"🧳 Maleta floja (C=5.0):    {ceros_flojo} coeficientes en cero")
print("Entre mas grande C, menos estricta es la maleta y menos caracteristicas se descartan.")
```
</details>


---
## ⚡ Resumen Relámpago

| Idea | En una frase |
|---|---|
| Clasificación multiclase | Decidir entre 3 o más categorías, no solo 2. |
| One-vs-Rest (OvR) | 3 jurados independientes, cada uno vota "es esta clase o no". |
| Softmax / Multinomial | Un solo comité que reparte 100% de probabilidad entre todas las clases a la vez. |
| Frontera de decisión | La línea (o curva) imaginaria donde el modelo cambia de opinión sobre la clase. |
| `PolynomialFeatures` | Le da al modelo "curvas extra" para separar patrones que una línea recta no puede. |
| Regularización L1 (Lasso) | Maleta estricta: descarta por completo las características poco útiles (coeficiente = 0). |
| Regularización L2 (Ridge) | Maleta flexible: reduce un poco todos los coeficientes, sin descartar ninguno. |

➡️ **Siguiente paso:** en el cuaderno [03 - Evaluación de Modelos y Métricas de Clasificación (Para Dummies)](03_Evaluacion_de_Modelos_y_Metricas_Clasificacion_Dummies.ipynb) aprenderás a medir qué tan bueno es realmente un modelo, más allá del simple "porcentaje de aciertos".


---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>
  </p>
</div>
